In [ ]:
/* 

    A2.E1 Motor

*/
select	count(*)
from	dlk.EXT_XtremePushResults   a 
where	`timestamp` between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and		campaign_name like 'Motor % TYR %' 
and interaction_type = 'sent' 
-- without the below conditions we get 70113 entries but it includes workflow/ sms / email so I think only 2/3 rd contacts sent out
and MessageType in ('SMS', 'EMAIL')  --> this brings down to 23058
-- and QuoteQueryGuid > ''  --> this brings down to 12148 these 2 conditions are a bit iffy as the volumes drop a lot
limit 100 

In [ ]:
/* 
    A3.B1.1 Motor 
-- Come back for this 
*/
select	count(distinct a.QuoteQueryGuid) 
from	stg.a1_motoracquisitionquoteinitiated a, 
        stg.a1_motoracquisitionquoteinitiated b 
Where	a.QuoteQueryGuid = b.RetrieveSessionToken
;

In [1]:
-- SCV Build
CREATE or Replace table stg.SCVBase_01 as 
select	SourceSystemReference, SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId 
from	ods.scv_customer_key 
where	SourceSystemId = 1 
;
CREATE or Replace table stg.SCVBase_02 as 
select	QuoteQueryGuid, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, c.PRN as AdditionalDriverFlag
from	ods.scv_customer_key				a,
		dlk.mfq_quotequery					b,
		dlk.mfq_quotedrivers				c
where	a.SourceSystemId = 2 
and		a.SourceSystemReference = c.QuoteDriverId
and		b.QuoteQueryId = c.QuoteQueryId
Group by QuoteQueryGuid, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, c.PRN 
;
CREATE or Replace table stg.SCVBase_03 as 
select	b.QuoteCodeReference, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId,
		case when a.SourceSystemReference like '%:JOINT' then 1 else 0 end as JointCustomerFlag 
from	ods.scv_customer_key				a,
		dlk.hfq_quotedetails				b
where	a.SourceSystemId = 3 
and		replace(a.SourceSystemReference, ':JOINT', '') = b.QuoteCodeReference
Group by b.QuoteCodeReference, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
;
CREATE or Replace table stg.SCVBase_04 as 
select	c.PolicyCode, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
from	ods.scv_customer_key				a,
		dlk.ext_home_qs_policyholderdetails	b,
		dlk.ext_home_qs_policydetails		c
where	a.SourceSystemId = 4 
and		a.SourceSystemReference = b.PolicyHolderId
and		b.PolicyId = c.PolicyId
;
CREATE or Replace table stg.SCVBase_05 as 
select	QuoteId, PolicyId, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId
from	ods.scv_customer_key				a,
		dlk.ext_travel_policy				b 
where	a.SourceSystemId = 5 
and		a.SourceSystemReference = b.MapfreCustomerId
Group by QuoteId, PolicyId, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId
;
CREATE or Replace table stg.SCVBase_06 as 
select	PK_ContactID, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, case when a.SourceSystemReference like '%:JOINT' then 1 else 0 end as JointCustomerFlag 
from	ods.scv_customer_key				a,
		dlk.life_tblcontacts				b 
where	a.SourceSystemId = 6 
and		replace(a.SourceSystemReference, ':JOINT', '') = b.PK_ContactID
Group by PK_ContactID, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, case when a.SourceSystemReference like '%:JOINT' then 1 else 0 end 
;
CREATE or Replace table stg.SCVBase_07 as 
select	RowKey, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
from	ods.scv_customer_key				a,
		dlk.vanquotedetails					b
where	a.SourceSystemId = 7 
and		a.SourceSystemReference = b.RowKey
Group by RowKey, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId
;
CREATE or Replace table stg.SCVBase_08 as 
select	quote_number, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId , user_id as email
from	ods.scv_customer_key				a,
		dlk.ext_travel_quotes				b
where	a.SourceSystemId = 8 
and		a.SourceSystemReference = b.quote_number
Group by quote_number, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId, user_id 
;
CREATE or Replace table stg.SCVBase_09 as 
select	Policy_Code, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
from	ods.scv_customer_key					a,
		dlk.ext_quotationstorage_policydetails	b
where	a.SourceSystemId = 9 
and		replace(a.SourceSystemReference, ':1', '') = b.Quotation_ID
Group by Policy_Code, a.SourceSystemReference, a.SCV_Customer_Key, a.ChillSourceCustomerKey, a.ChillSourceAddressKey, a.SourceSystemId 
;

StatementMeta(, 719a678f-3be3-4d5e-a76f-0d4b928f6432, 10, Finished, Available, Finished, True)

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
CREATE or Replace table ods.SCVBase as 
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, cast(null  as varchar(225)) as QuoteReference, cast(1 as VARCHAR(10)) AdditionalCustomerFlag  
from stg.SCVBase_01 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, QuoteQueryGuid as QuoteReference, AdditionalDriverFlag   
from stg.SCVBase_02 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, QuoteCodeReference, 1    
from stg.SCVBase_03 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, null as ClientCode, PolicyCode, SourceSystemReference as QuoteReference, 1    
from stg.SCVBase_04 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode, CAST(PolicyId as varchar(225)) as PolicyCode, CAST(QuoteId as varchar(225)) as QuoteReference, CAST(1 AS VARCHAR(10))   
from stg.SCVBase_05 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(PK_ContactID as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, cast(null  as varchar(225)) as QuoteReference, CAST(1 + JointCustomerFlag AS VARCHAR(10))   
from stg.SCVBase_06 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, cast(SourceSystemReference as varchar(225)) as ClientCode,cast(null as varchar(225)) as PolicyCode, RowKey as QuoteReference, CAST(1 AS VARCHAR(10))  
from stg.SCVBase_07 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, email as ClientCode,cast(null as varchar(225)) as PolicyCode, cast(quote_number as varchar(225)) as QuoteReference, CAST(1 AS VARCHAR(10)) 
from stg.SCVBase_08 union
select SCV_Customer_Key, ChillSourceCustomerKey, ChillSourceAddressKey, SourceSystemId, SourceSystemReference, left(PolicyCode,6) as ClientCode, cast(Policy_Code as varchar(225)), left(SourceSystemReference, 36) as QuoteReference, CAST(1 AS VARCHAR(10))  
from stg.SCVBase_09
;

StatementMeta(, 719a678f-3be3-4d5e-a76f-0d4b928f6432, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
select count(*) from ods.SCVBase ;

StatementMeta(, 719a678f-3be3-4d5e-a76f-0d4b928f6432, 12, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [5]:
Create or Replace Table stg.RelayCustomers as 
select	SCV_Customer_Key, ChillSourceCustomerKey, ClientCode
from	ods.scvbase			a 
Where	a.SourceSystemId = 1 
and		len(a.ClientCode) = 6
Group by SCV_Customer_Key, ChillSourceCustomerKey, ClientCode
;

StatementMeta(, 6164cb6d-4e33-4f70-80a6-75ccb5869c66, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [11]:
Create or Replace Table ods.ScvEadmLink as 
select	case when b.SCV_Customer_Key < 10000000000000 then coalesce(b.SCV_Customer_Key, a.SCV_Customer_Key) else a.SCV_Customer_Key end as SCV_Customer_Key_Final,
        a.*
from	ods.scvbase			a 
left join 
        stg.RelayCustomers  b 
            on  b.ClientCode = left(a.PolicyCode,6)
            and	a.SourceSystemId in (4,9)
            and	len(a.PolicyCode) = 9
;

StatementMeta(, 6164cb6d-4e33-4f70-80a6-75ccb5869c66, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [ ]:
 select * 
 from   ods.ScvEadmLink 
 where  SCV_Customer_Key_Final <> SCV_Customer_Key 
 /*and    SCV_Customer_Key_Final > 10000000000000*/
 limit 100 

StatementMeta(, 6164cb6d-4e33-4f70-80a6-75ccb5869c66, 11, Finished, Available, Finished, False)

<Spark SQL result set with 100 rows and 10 fields>

In [ ]:
--- 
/*
Motor	Renewal	R0.B1
Motor	Renewal	RO.F1
*/

select  top 10 * 
from    ods.eventstream 
where   PolicyTypeGroup = 'Motor' 
and     EventSourceId = 3 
and     EventDescription like 'Renewal Offer % Held %'  

In [ ]:
/*
Motor	Renewal	R1.B2
*/
select  count(distinct PolicyCode)
from    edw.tbl_fact_policy_renewals a      
Where   a.LYPolicyTypeGroup = 'Motor' 
and     a.LYPolicyRenewDateAdj between '2026-07-01' and '2026-07-31' 
and     PolicyOfferNum = 1 and RenEURPremInvite <> RenEURPremOffer
;


In [ ]:
-- IGNORE HM2 as no data 
-- Home	Mta	HM2 -- No Data -- Ignore
-- Home	Mta	HM4.F1  -- No data for that 
-- Home	Mta	HM5 -- No data for that 

In [ ]:
/* 
Home	Mta	HM4.B1.6
*/
-- use the MTA base and join with MyChillWorkflow_DocumentStatus

select  isAccepted, count(distinct b.PolicyCode)
from    (
                select  b.PolicyCode
                from    stg.JJulyMTAs a, stg.JulyPolicyState b
                Where   a.PolicyCode = b.PolicyCode
                and     PolicyTypeGroup = 'Home'
                and     PolicyStatusDesc not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term' , 'Lapsed for Transfer') 
            )  a, dlk.MyChillWorkflow_DocumentStatus  b 
Where   a.PolicyCode = b.PolicyCode
and     `Timestamp` between '2026-07-01 00:00:00.000' AND '2026-08-01 00:00:00.000' 
/*and     isAccepted IsRejected */
Group by isAccepted 
;

In [ ]:
-- Home	Mta	HM4.F1
 -- No data for that 

In [ ]:
 /*  Home	MTA	HM4a -- mark as not available 
*/
--select  Count(distinct b.PolicyCode), sum(CCYGrossPremium), sum(CCYFees)
--from    stg.JJulyMTAs a, stg.JulyPolicyState b 
--Where   a.PolicyCode = b.PolicyCode
--and     PolicyTypeGroup = 'Home'
--and     PolicyStatusDesc not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term' , 'Lapsed for Transfer') 
--and     ( 
--            CCYGrossPremium <> 0 
--        or 
--            CCYFees <> 0 )
--and     MTAGrossNum > 0 -- number from recon to be modified -- not 275            

In [ ]:
/*
    Home	MTA	HM6.B1.2 - counts follow in the next cell 
*/
SELECT  count(distinct a.PolicyCode)
FROM    (
            select  b.PolicyCode
            from    stg.JJulyMTAs a, stg.JulyPolicyState b
            Where   a.PolicyCode = b.PolicyCode
            and     PolicyTypeGroup = 'Home'
            and     PolicyStatusDesc not in ('Cancelled', 'Lapsed', 'Cancelled Mid Term' , 'Lapsed for Transfer') 
        )                                       a,
        ods.EventStream                          d 
Where   d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Home'
and     d.EventDateTime between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000'  
and     (   EventDescription like '%Emailed Document%' 
        or 
            EventDescription like '%Document Transmitted%'
        )
and     EventDescription in (

        'New Business - Emailed Document - Terms Of Business',
        'New Business - Emailed Document - Cover Letter',
        'New Business - Emailed Document - 04 - Suitability Statement',
        'New Business - Document Transmitted - 04 - Suitability Statement',
        'New Business - Document Transmitted - Terms Of Business',
        'New Business - Document Transmitted - Cover Letter'
)        
and     a.PolicyCode = d.SourcePolicyReference  

In [ ]:
/*
Home	Renewal	HR2.B1.2.F1
*/

select count(distinct a.PolicyCode), count(distinct a.QuoteCodeReference)
from    stg.HomeRenewalsDoingHFQ    a 
left join 
        (select QuoteCodeReference  
        from    dlk.HFQ_Response_Quotes  
        where   Quotes_Premium <> 0 
        and     Quotes_Outcome = 'PremiumReturned'        
        Group by QuoteCodeReference) b
        on a.QuoteCodeReference = b.QuoteCodeReference 
Where   b.QuoteCodeReference is null 

In [ ]:
/* Home	Renewal HR4.B2.4
 */
select  count(distinct a.PolicyCode) 
from    edw.EXP_MyChill_Chase_Daily_Snapshot_Home a 
Where	coalesce(try_cast(concat(right(SaleDate,4), '-',substring(SaleDate,4,2),'-',left(SaleDate,2)) as date),'1900-01-01') between '2026-05-01' and '2026-07-31' 
and     PolicyTypeGroup = 'Home' 
and case 
        When Comp_DDM_Status				= 'O' then 1 
        When Gap_In_Cov_Ltr_Status			= 'O' then 1 
        When Val_For_Spec_Item_Status		= 'O' then 1 
        When PPS_Num_Status					= 'O' then 1 
        When Identification_Status			= 'O' then 1 
        When Digital_Journey_Status			= 'O' then 1 
        When Finance_Form_Status			= 'O' then 1 
        else 0
    End = 1 
and     Campaign = 'DAY 1' 
and     PolicyType = 'Renewals' 

In [ ]:
/*
Travel	Renewal	TR.O1 -- this is not 10106 -- this is 2130
        The 10106 is the total renewal + new business volume -- SQL09 reports them together and cannot split .. but we can repotr separately 
*/
SELECT
    COUNT(*) AS TotalSales
FROM dlk.EXT_Travel_Policy p
WHERE p.PurchaseDate >= '2026-07-01'
  AND p.PurchaseDate <  '2026-08-01'
  AND p.RenewalFlag = 1
;

In [ ]:
/*
Travel	Renewal	TR1.E1
*/
select	distinct email
from	dlk.ext_xtremepushresults_policy
where	campaign_name in 
					( 
						'Travel - Renewal Date in the next 14 Days',
						'Travel - Renewal Date in the next  7 Days' 
					) 
and		[timestamp] between '2026-07-01 00:00:00.000000' and '2026-08-01 00:00:00.000000' 
and		MessageType in ('EMAIL', 'SMS') 

In [ ]:

/* 
Van	Acquisition	VA5.B1.4
*/
SELECT  count(distinct a.PolicyCode), count(*), min(EventDateTime), max(EventDateTime)
FROM    stg.VanSalesJuly                         a,
        ods.EventStream                          d 
Where   a.PolicyCode = d.SourcePolicyReference
and     d.EventSourceId = 3 
and     d.PolicyTypeGroup = 'Van'
and     EventDateTime >= '2026-07-01 00:00:00.000' 
and     EventDateTime <= '2026-08-01 00:00:00.000' 
and     ReportingSaleType = 'New Business' 
and     EventDescription in 
        ( 
        'Document Chase - Emailed Document - PC Chase Email'
        )



StatementMeta(, efdfc7ab-03e1-4ea7-926a-075236abf1da, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

In [ ]:
/*
Van	Arrears	VARR.O2
*/

select  count(distinct SourcePolicyReference) 
from            (select  distinct SourcePolicyReference 
                from    ods.eventstream        a 
                Where    EventDateTime between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
                and     EventSourceId = 3 
                and     PolicyTypeGroup = 'Van'
                and     EventDescription like 'Insurer Led%') a,   
                (select  distinct PolicyCode
                from    dlk.EXT_XtremePushResults  
                where   upper(campaign_name) like '%RREARS%' 
                and     timestamp between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
                and     MessageType in ('SMS', 'EMAIL')
                UNION
                select  distinct PolicyCode
                from    dlk.EXT_XtremePushResults_Policy  
                where   upper(campaign_name) like '%RREARS%' 
                and     timestamp between '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
                and     MessageType in ('SMS', 'EMAIL')
                                ) b 
Where   a.SourcePolicyReference = b.PolicyCode




StatementMeta(, efdfc7ab-03e1-4ea7-926a-075236abf1da, 8, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [ ]:
/*
Van	Cancellation	VC6
*/
select  EventDescription, count(distinct  SourcePolicyReference )
from    ods.eventstream        a 
Where    EventDateTime between  '2026-07-01 00:00:00.000' and '2026-08-01 00:00:00.000' 
and     EventSourceId = 3 
and     PolicyTypeGroup = 'Van'
and     EventDescription = 'Insurer Led Cancelation - Emailed Document - Reg canx email template'
Group by  EventDescription

StatementMeta(, efdfc7ab-03e1-4ea7-926a-075236abf1da, 13, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [15]:
/*
Van	Claims	VCL1a
*/
-- step 1 
Create or Replace Table stg.Claims_genesys_derived_data_filtered as 
select	a.ConversationId, CustomerPhoneNumber,a.sessionIndex,A.conversationStartTime
from	stg.A0_genesys_derived_data	a
where	queueName = 'INBOUND_Claims'
;


StatementMeta(, efdfc7ab-03e1-4ea7-926a-075236abf1da, 14, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [18]:
/*
Van	Claims	VCL1a
*/
-- step 2 
Create or Replace Table stg.Van_PhoneNumbers as 
select	PolicyCode, ClientCode, CustomerPhone, RapierCustomerId
FROM	pii.customer_data_assembled a,
        (
            select	ClientCode, PolicyCode
            from	edw.tbl_fact_policy_mvt
            Where	PolicyTypeGroup = 'Van' 
            and		EffectiveDate = '2026-07-31' 
            and		OpenNum = 1 
            Group by ClientCode, PolicyCode
        )   b
Where	SourceSystemCustomerId = ClientCode
and     SourceSystemId = 1 			
;                

StatementMeta(, efdfc7ab-03e1-4ea7-926a-075236abf1da, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [20]:
/*
Van	Claims	VCL1a
*/
-- result
select  count(distinct ConversationId), count(distinct PolicyCode) 
from    stg.Van_PhoneNumbers a, stg.Claims_genesys_derived_data_filtered b 
Where   b.CustomerPhoneNumber = a.CustomerPhone

StatementMeta(, efdfc7ab-03e1-4ea7-926a-075236abf1da, 19, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

In [21]:
/*
Van	Doc Request	VD.V1
*/
-- step 1 
Create or Replace Table stg.DocRequest_genesys_derived_data_filtered as 
select	a.ConversationId, CustomerPhoneNumber,a.sessionIndex,A.conversationStartTime
from	stg.A0_genesys_derived_data	a
where	queueName = 'INBOUND_Documents_Out'
;


StatementMeta(, efdfc7ab-03e1-4ea7-926a-075236abf1da, 20, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [27]:
/*
Van	Doc Request	VD.V1
*/
-- result
select  count(distinct SourcePolicyReference)
from    ods.EventStream a,
        (select  distinct PolicyCode , conversationStartTime
        from    stg.Van_PhoneNumbers a, stg.DocRequest_genesys_derived_data_filtered b 
        Where   b.CustomerPhoneNumber = a.CustomerPhone) b 
Where   a.SourcePolicyReference = b.PolicyCode
and     a.EventDateTime between conversationStartTime and dateadd(day,2,conversationStartTime)
and     EventDescription in (
            'Prior Year Quotes - Emailed Document - C&D Email'
            ) 

StatementMeta(, efdfc7ab-03e1-4ea7-926a-075236abf1da, 26, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>

In [28]:
/*
Van	Renewal	VR3b.F1
*/
SELECT COUNT(DISTINCT ConversationId) FROM (
            select	a.ConversationId
            from	stg.A0_genesys_derived_data	a
            where	a.queueName = 'INBOUND_Van_Renewals'  
            Group by ConversationId
            HAVING sum(abandoned) > 0 ) X 

StatementMeta(, efdfc7ab-03e1-4ea7-926a-075236abf1da, 27, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 1 fields>